# Notebook 04 — Explainable AI and Synthetic-Ground-Truth Validation (Reviewer-Response Version)

This notebook explains the selected XGBoost model on held-out synthetic records and adds the reviewer-requested **ground-truth explanation check**.

In addition to comparing permutation importance with SHAP, it compares both model-explanation rankings against the known synthetic generator by perturbing one input feature at a time and measuring the resulting change in the generator's **noise-free probability surface**. The notebook also reports each feature's direct target term and interaction-rule participation.

This establishes internal explanation fidelity to the known simulation mechanism. It does **not** establish causality or real-world validity.


In [ ]:
# =========================
# 0. Setup and Reproducibility
# =========================
import warnings
warnings.filterwarnings("ignore")

import json
import platform
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import sklearn

SEED = 42
np.random.seed(SEED)

PERMUTATION_SAMPLE_N = 5000
PERMUTATION_REPEATS = 10
SHAP_SAMPLE_N = 1000

# Repository-root resolver: allows notebooks to run from the repository root
# or from the notebooks/ directory without changing output paths.
def _find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".project-root").exists():
            return candidate
    return current

REPO_ROOT = _find_repo_root()
PROJECT_DIR = REPO_ROOT / "project_delay_outputs"
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
PLOTS_DIR = RESULTS_DIR / "plots"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

print("XAI seed:", SEED)
print("Permutation sample size:", PERMUTATION_SAMPLE_N)
print("SHAP sample size:", SHAP_SAMPLE_N)


In [ ]:
# =========================
# 1. Load Data, Selected Model, and Metadata
# =========================
data_path = DATA_DIR / "synthetic_project_delay_dataset.csv"
model_path = MODEL_DIR / "best_delay_prediction_pipeline.pkl"
metadata_path = MODEL_DIR / "model_metadata.pkl"

if not data_path.exists():
    raise FileNotFoundError("Run Notebook 01 first to generate the synthetic dataset.")

if not metadata_path.exists():
    raise FileNotFoundError("Run Notebook 02 first to create model metadata.")

metadata = joblib.load(metadata_path)

best_model_name = metadata["best_model_name"]
features = metadata["features"]
categorical_features = metadata["categorical_features"]
numeric_features = metadata["numeric_features"]
target = metadata["target"]

if not model_path.exists():
    raise FileNotFoundError(
        "The selected best-model sklearn pipeline was not found. "
        "This XAI notebook expects the saved sklearn pipeline from Notebook 02. "
        f"Notebook 02 reports the selected model as: {best_model_name}"
    )

df = pd.read_csv(data_path)
model = joblib.load(model_path)

if not hasattr(model, "named_steps"):
    raise TypeError(
        "The loaded best model is not an sklearn Pipeline. "
        "This notebook is designed for the selected tree/classical pipeline."
    )

X = df[features].copy()
y = df[target].astype(int)

print("Loaded model:", best_model_name)
print("Dataset shape:", df.shape)
print("Model-input features:", len(features))
print("Interpretation scope:", metadata.get(
    "interpretation_scope",
    "Synthetic proof-of-concept environment; real-world validation is required."
))


## 2. Held-Out Test Set and Result Consistency

The model is explained using the same stratified 80/20 partition used for the baseline experiment in Notebook 02. XAI is therefore calculated on **held-out test records**, rather than on the full dataset that includes training observations.

The direct test-set metrics are also compared with `FINAL_baseline_model_comparison.csv` when that file is available. This provides an explicit consistency check so that model-performance values used by the XAI notebook do not silently diverge from the final Results table.


In [ ]:
# =========================
# 2.1 Recreate the Baseline Held-Out Test Partition
# =========================
baseline_seed = int(metadata.get("baseline_seed", SEED))
test_fraction = float(metadata.get("baseline_test_fraction", 0.20))

_, X_test, _, y_test = train_test_split(
    X,
    y,
    test_size=test_fraction,
    random_state=baseline_seed,
    stratify=y
)

def positive_class_probability(fitted_model, frame):
    proba = fitted_model.predict_proba(frame)

    if proba.ndim == 2 and proba.shape[1] >= 2:
        return proba[:, 1]

    return np.asarray(proba).reshape(-1)

y_pred_test = model.predict(X_test)
y_proba_test = positive_class_probability(model, X_test)

direct_metrics = {
    "Model": best_model_name,
    "Accuracy": accuracy_score(y_test, y_pred_test),
    "Precision": precision_score(y_test, y_pred_test, zero_division=0),
    "Recall": recall_score(y_test, y_pred_test, zero_division=0),
    "F1_Score": f1_score(y_test, y_pred_test, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, y_proba_test)
}

direct_metrics_df = pd.DataFrame([direct_metrics])
display(direct_metrics_df)

baseline_results_path = RESULTS_DIR / "FINAL_baseline_model_comparison.csv"
consistency_rows = []

if baseline_results_path.exists():
    baseline_results_df = pd.read_csv(baseline_results_path)
    reported_row = baseline_results_df[
        baseline_results_df["Model"].astype(str) == str(best_model_name)
    ]

    if len(reported_row) == 1:
        reported_row = reported_row.iloc[0]

        for metric in ["Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC"]:
            direct_value = float(direct_metrics[metric])
            reported_value = float(reported_row[metric])

            consistency_rows.append({
                "Metric": metric,
                "Direct_XAI_Notebook_Value": direct_value,
                "FINAL_Model_Table_Value": reported_value,
                "Absolute_Difference": abs(direct_value - reported_value)
            })

        consistency_df = pd.DataFrame(consistency_rows)
        display(consistency_df)

        max_difference = consistency_df["Absolute_Difference"].max()

        if max_difference <= 1e-10:
            print("Baseline-result consistency check: PASSED")
        else:
            print(
                "WARNING: baseline-result consistency check found a difference. "
                "Use the FINAL model table as the manuscript source of truth and "
                "investigate before reporting XAI results."
            )
    else:
        consistency_df = pd.DataFrame()
        print("Selected model was not uniquely found in the FINAL baseline model table.")
else:
    consistency_df = pd.DataFrame()
    print("FINAL baseline comparison file not found; direct metrics were calculated only.")

direct_metrics_df.to_csv(
    RESULTS_DIR / "FINAL_xai_holdout_model_metrics.csv",
    index=False
)

consistency_df.to_csv(
    RESULTS_DIR / "FINAL_xai_holdout_consistency_check.csv",
    index=False
)


## 3. Global Permutation Importance

Permutation importance measures the decrease in held-out F1-score when one original input feature is randomly shuffled while all other feature values are retained. A larger decrease indicates greater reliance of the trained model on that feature **within the synthetic test environment**.

The analysis does not imply that changing the feature would causally change project delay.


In [ ]:
# =========================
# 3.1 Held-Out Permutation Importance
# =========================
perm_sample_n = min(PERMUTATION_SAMPLE_N, len(X_test))

X_perm = X_test.sample(
    n=perm_sample_n,
    random_state=SEED
)
y_perm = y_test.loc[X_perm.index]

perm = permutation_importance(
    model,
    X_perm,
    y_perm,
    n_repeats=PERMUTATION_REPEATS,
    random_state=SEED,
    scoring="f1",
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "Feature": features,
    "Importance_Mean": perm.importances_mean,
    "Importance_Std": perm.importances_std
}).sort_values(
    "Importance_Mean",
    ascending=False
).reset_index(drop=True)

positive_importance = perm_df["Importance_Mean"].clip(lower=0)
positive_total = positive_importance.sum()

if positive_total > 0:
    perm_df["Importance_Percentage_Positive_Total"] = (
        positive_importance / positive_total * 100
    )
else:
    perm_df["Importance_Percentage_Positive_Total"] = 0.0

perm_df["Rank"] = np.arange(1, len(perm_df) + 1)

display(perm_df.head(25))

plot_df = perm_df.head(20).sort_values("Importance_Mean")

ax = plot_df.plot(
    x="Feature",
    y="Importance_Mean",
    kind="barh",
    figsize=(10, 7),
    legend=False
)

ax.set_title("Held-Out Permutation Feature Importance")
ax.set_xlabel("Mean decrease in F1-score after permutation")

plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "FINAL_xai_permutation_importance.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

perm_df.to_csv(
    RESULTS_DIR / "FINAL_xai_global_permutation_importance.csv",
    index=False
)


## 4. Model-Native Feature Importance

For tree-based models, the estimator's internal feature-importance values are extracted after preprocessing. One-hot encoded categorical levels are aggregated back to their original project feature.

Model-native importance is included as a supplementary interpretation method. It is not used as evidence of causal importance.


In [ ]:
# =========================
# 4.1 Model-Native Importance and Feature Aggregation
# =========================
def get_transformed_feature_names(fitted_pipeline):
    preprocessor = fitted_pipeline.named_steps["preprocessor"]

    try:
        names = preprocessor.get_feature_names_out().tolist()
        cleaned = []

        for name in names:
            text = str(name)

            if "__" in text:
                text = text.split("__", 1)[1]

            cleaned.append(text)

        return cleaned

    except Exception:
        names = list(numeric_features)

        if len(categorical_features) > 0:
            onehot = preprocessor.named_transformers_["cat"].named_steps["onehot"]
            names.extend(
                onehot.get_feature_names_out(categorical_features).tolist()
            )

        return names


def map_to_main_feature(transformed_name):
    text = str(transformed_name)

    if "__" in text:
        text = text.split("__", 1)[1]

    for cat in categorical_features:
        if text == cat or text.startswith(cat + "_"):
            return cat

    return text


model_step = model.named_steps["model"]
transformed_names = get_transformed_feature_names(model)

native_df = pd.DataFrame()
grouped_native_df = pd.DataFrame()

if hasattr(model_step, "feature_importances_"):
    if len(transformed_names) != len(model_step.feature_importances_):
        raise ValueError(
            "Number of transformed feature names does not match the model's "
            "feature_importances_ vector."
        )

    native_df = pd.DataFrame({
        "Transformed_Feature": transformed_names,
        "Main_Feature": [
            map_to_main_feature(name)
            for name in transformed_names
        ],
        "Importance": model_step.feature_importances_
    }).sort_values(
        "Importance",
        ascending=False
    ).reset_index(drop=True)

    grouped_native_df = (
        native_df
        .groupby("Main_Feature", as_index=False)["Importance"]
        .sum()
        .sort_values("Importance", ascending=False)
        .reset_index(drop=True)
    )

    grouped_native_df["Rank"] = np.arange(
        1,
        len(grouped_native_df) + 1
    )

    display(native_df.head(30))
    display(grouped_native_df.head(20))

    plot_native = grouped_native_df.head(20).sort_values("Importance")

    ax = plot_native.plot(
        x="Main_Feature",
        y="Importance",
        kind="barh",
        figsize=(10, 7),
        legend=False
    )

    ax.set_title("Grouped Model-Native Feature Importance")
    ax.set_xlabel("Aggregated model-native importance")

    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR / "FINAL_xai_model_native_importance.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    native_df.to_csv(
        RESULTS_DIR / "FINAL_xai_native_transformed_importance.csv",
        index=False
    )

    grouped_native_df.to_csv(
        RESULTS_DIR / "FINAL_xai_native_grouped_importance.csv",
        index=False
    )

else:
    print(
        "The selected model does not expose feature_importances_. "
        "Permutation importance and SHAP remain the primary global XAI methods."
    )


## 5. Global SHAP Explanation

SHAP values describe how the selected model distributes its prediction across the transformed input features. For publication-level interpretation, the absolute SHAP values of one-hot encoded levels are aggregated back to the original project-management features.

The reported SHAP ranking should be described as **model influence within the synthetic environment**, not as a causal ranking of real-world project-delay drivers.


In [ ]:
# =========================
# 5.1 SHAP Global Importance
# =========================
shap_available = False
shap_df = pd.DataFrame()
grouped_shap_df = pd.DataFrame()
shap_explainer = None
shap_version = None

def normalize_shap_array(values):
    """
    Convert common SHAP binary-classification output formats into
    an n_samples x n_features array for the positive class.
    """
    if isinstance(values, list):
        if len(values) == 2:
            arr = np.asarray(values[1])
        else:
            arr = np.asarray(values[0])
    elif hasattr(values, "values"):
        arr = np.asarray(values.values)
    else:
        arr = np.asarray(values)

    if arr.ndim == 3:
        if arr.shape[-1] == 2:
            arr = arr[:, :, 1]
        elif arr.shape[0] == 2:
            arr = arr[1]
        else:
            raise ValueError(
                f"Unexpected 3D SHAP output shape: {arr.shape}"
            )

    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    return arr


try:
    import shap

    shap_version = shap.__version__

    shap_sample_n = min(SHAP_SAMPLE_N, len(X_test))

    X_shap_raw = X_test.sample(
        n=shap_sample_n,
        random_state=SEED
    )

    X_shap_transformed = (
        model.named_steps["preprocessor"]
        .transform(X_shap_raw)
    )

    if hasattr(X_shap_transformed, "toarray"):
        X_shap_transformed = X_shap_transformed.toarray()

    if hasattr(model_step, "feature_importances_"):
        shap_explainer = shap.TreeExplainer(model_step)
        raw_shap_values = shap_explainer.shap_values(
            X_shap_transformed
        )
        shap_arr = normalize_shap_array(raw_shap_values)
    else:
        background_n = min(100, X_shap_transformed.shape[0])
        background = shap.sample(
            X_shap_transformed,
            background_n,
            random_state=SEED
        )

        shap_explainer = shap.Explainer(
            model_step.predict_proba,
            background
        )

        raw_explanation = shap_explainer(
            X_shap_transformed
        )

        shap_arr = normalize_shap_array(raw_explanation)

    if shap_arr.shape[1] != len(transformed_names):
        raise ValueError(
            "SHAP feature dimension does not match transformed feature names: "
            f"{shap_arr.shape[1]} vs {len(transformed_names)}"
        )

    shap_df = pd.DataFrame({
        "Transformed_Feature": transformed_names,
        "Main_Feature": [
            map_to_main_feature(name)
            for name in transformed_names
        ],
        "Mean_ABS_SHAP": np.abs(shap_arr).mean(axis=0),
        "Mean_Signed_SHAP": shap_arr.mean(axis=0)
    }).sort_values(
        "Mean_ABS_SHAP",
        ascending=False
    ).reset_index(drop=True)

    grouped_shap_df = (
        shap_df
        .groupby("Main_Feature", as_index=False)
        .agg(
            Mean_ABS_SHAP=("Mean_ABS_SHAP", "sum"),
            Mean_Signed_SHAP=("Mean_Signed_SHAP", "sum")
        )
        .sort_values("Mean_ABS_SHAP", ascending=False)
        .reset_index(drop=True)
    )

    grouped_shap_df["Rank"] = np.arange(
        1,
        len(grouped_shap_df) + 1
    )

    display(shap_df.head(30))
    display(grouped_shap_df.head(20))

    plot_shap = grouped_shap_df.head(20).sort_values(
        "Mean_ABS_SHAP"
    )

    ax = plot_shap.plot(
        x="Main_Feature",
        y="Mean_ABS_SHAP",
        kind="barh",
        figsize=(10, 7),
        legend=False
    )

    ax.set_title("Grouped SHAP Importance on Held-Out Synthetic Projects")
    ax.set_xlabel("Mean absolute SHAP value")

    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR / "FINAL_xai_shap_grouped_importance.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    shap_df.to_csv(
        RESULTS_DIR / "FINAL_xai_shap_transformed_importance.csv",
        index=False
    )

    grouped_shap_df.to_csv(
        RESULTS_DIR / "FINAL_xai_shap_grouped_importance.csv",
        index=False
    )

    shap_available = True

except Exception as e:
    print("SHAP analysis skipped because of an environment/model compatibility issue:")
    print(type(e).__name__, "-", e)
    print(
        "Permutation importance remains available. "
        "Do not report SHAP results unless this section completes successfully."
    )


## 6. Agreement Between Global XAI Methods

When SHAP is available, the permutation and SHAP rankings are compared quantitatively. This provides a transparent basis for statements that the two explanation methods identify similar model-sensitive features.

Agreement between methods still reflects **the trained model and synthetic generator assumptions**, not independent confirmation of real-world causality.


In [ ]:
# =========================
# 6.1 Permutation-SHAP Ranking Agreement
# =========================
xai_agreement_df = pd.DataFrame()
xai_agreement_summary_df = pd.DataFrame()

if shap_available and not grouped_shap_df.empty:
    xai_agreement_df = (
        perm_df[
            ["Feature", "Importance_Mean"]
        ]
        .rename(columns={
            "Feature": "Main_Feature",
            "Importance_Mean": "Permutation_Importance"
        })
        .merge(
            grouped_shap_df[
                ["Main_Feature", "Mean_ABS_SHAP"]
            ],
            on="Main_Feature",
            how="inner"
        )
    )

    xai_agreement_df["Permutation_Rank"] = (
        xai_agreement_df["Permutation_Importance"]
        .rank(ascending=False, method="average")
    )

    xai_agreement_df["SHAP_Rank"] = (
        xai_agreement_df["Mean_ABS_SHAP"]
        .rank(ascending=False, method="average")
    )

    # Pearson correlation of rank values is the Spearman rank correlation.
    rank_correlation = xai_agreement_df[
        ["Permutation_Rank", "SHAP_Rank"]
    ].corr(method="pearson").iloc[0, 1]

    top_k = min(10, len(xai_agreement_df))

    top_perm = set(
        xai_agreement_df
        .nsmallest(top_k, "Permutation_Rank")["Main_Feature"]
    )

    top_shap = set(
        xai_agreement_df
        .nsmallest(top_k, "SHAP_Rank")["Main_Feature"]
    )

    overlap_count = len(top_perm.intersection(top_shap))
    overlap_percent = 100 * overlap_count / top_k if top_k else np.nan

    xai_agreement_summary_df = pd.DataFrame([{
        "Compared_Features": len(xai_agreement_df),
        "Spearman_Rank_Correlation": rank_correlation,
        "Top_K": top_k,
        "Top_K_Overlap_Count": overlap_count,
        "Top_K_Overlap_Percent": overlap_percent
    }])

    display(
        xai_agreement_df.sort_values(
            "Permutation_Rank"
        ).head(20)
    )

    display(xai_agreement_summary_df)

    xai_agreement_df.to_csv(
        RESULTS_DIR / "FINAL_xai_permutation_shap_agreement.csv",
        index=False
    )

    xai_agreement_summary_df.to_csv(
        RESULTS_DIR / "FINAL_xai_permutation_shap_agreement_summary.csv",
        index=False
    )

else:
    print("SHAP is unavailable; permutation-SHAP agreement was not calculated.")


In [ ]:
# =========================
# 6.2 Compare XAI With the Known Synthetic Generating Mechanism
# =========================
# Ground-truth importance is defined here as the mean absolute change in the
# NOISE-FREE generator probability after permuting one original feature.
# This captures direct coefficients, nonlinear transforms, and interaction
# rules in a way that is comparable with permutation/SHAP rankings.
#
# It is a property of the synthetic generator, not an empirical causal effect.

def generator_rule_score(frame):
    rs = pd.Series(0.0, index=frame.index)
    rs += np.where(
        (frame["Authorities_Involved"] > 4)
        & (frame["Approval_Duration_Days"] > 35), 0.85, 0
    )
    rs += np.where(
        (frame["Design_Change_Count"] >= 5)
        & (frame["Scope_Clarity_Score"] < 6), 0.90, 0
    )
    rs += np.where(
        (frame["Stakeholder_Communication_Score"] < 5)
        & (frame["Coordination_Score"] < 5), 0.80, 0
    )
    rs += np.where(
        (frame["Resource_Availability_Score"] < 5)
        | (frame["Labor_Productivity_Score"] < 5), 0.65, 0
    )
    rs += np.where(
        (frame["Supplier_Reliability_Score"] < 5)
        & (frame["Procurement_Lead_Time_Days"] > 45), 0.70, 0
    )
    rs += np.where(
        (frame["Contractor_Financial_Stability"] < 5)
        & (frame["Payment_Delay_Days"] > 20), 0.75, 0
    )
    rs += np.where(
        (frame["External_Risk_Score"] > 7)
        | (frame["Force_Majeure_Flag"] == 1), 0.95, 0
    )
    rs += np.where(
        (frame["Schedule_Buffer_Percent"] < 5)
        & (frame["Complexity_Score"] >= 3), 0.65, 0
    )
    rs -= np.where(
        (frame["BIM_Adoption_Level"] >= 2)
        & (frame["Coordination_Score"] >= 7), 0.45, 0
    )
    rs -= np.where(
        (frame["AI_Tools_Adoption_Level"] >= 2)
        & (frame["Project_Manager_Experience_Years"] >= 8), 0.35, 0
    )
    rs -= np.where(
        (frame["Schedule_Buffer_Percent"] >= 12)
        & (frame["Scope_Clarity_Score"] >= 7), 0.35, 0
    )
    return pd.Series(rs, index=frame.index)


def generator_noise_free_probability(frame):
    rs = generator_rule_score(frame)

    z = (
        -2.40
        + 0.34 * frame["Complexity_Score"]
        + 0.015 * (frame["Planned_Duration_Days"] / 10)
        + 0.020 * np.log1p(frame["Planned_Budget_Million"])
        + 0.10 * frame["Design_Change_Count"]
        + 0.09 * frame["Change_Request_Count"]
        + 0.012 * frame["Approval_Duration_Days"]
        + 0.012 * frame["Procurement_Lead_Time_Days"]
        + 0.018 * frame["Payment_Delay_Days"]
        + 0.11 * frame["External_Risk_Score"]
        + 0.06 * frame["Quality_Defect_Rate"]
        + 0.18 * frame["Safety_Incident_Count"]
        - 0.12 * frame["Scope_Clarity_Score"]
        - 0.10 * frame["Stakeholder_Communication_Score"]
        - 0.09 * frame["Coordination_Score"]
        - 0.08 * frame["Contractor_Experience_Score"]
        - 0.08 * frame["Resource_Availability_Score"]
        - 0.07 * frame["Labor_Productivity_Score"]
        - 0.06 * frame["Equipment_Availability_Score"]
        - 0.06 * frame["Supplier_Reliability_Score"]
        - 0.06 * frame["Contractor_Financial_Stability"]
        - 0.05 * frame["Owner_Decision_Speed_Score"]
        - 0.035 * frame["Schedule_Buffer_Percent"]
        - 0.18 * frame["BIM_Adoption_Level"]
        - 0.12 * frame["AI_Tools_Adoption_Level"]
        + rs
    )
    return 1.0 / (1.0 + np.exp(-z))


def preserve_generator_dependencies(frame, changed_feature):
    """
    Recompute only deterministic derived fields whose source feature was
    perturbed. This avoids leaving impossible approval/external-risk composites.
    """
    adjusted = frame.copy()

    if changed_feature in {
        "Authorities_Involved",
        "Approval_Per_Authority_Days",
    }:
        adjusted["Approval_Duration_Days"] = (
            adjusted["Authorities_Involved"]
            * adjusted["Approval_Per_Authority_Days"]
        )

    if changed_feature in {
        "Weather_Risk_Score",
        "Permit_Risk_Score",
        "Inflation_Risk_Score",
        "Site_Condition_Risk_Score",
        "Force_Majeure_Flag",
    }:
        adjusted["External_Risk_Score"] = (
            0.27 * adjusted["Weather_Risk_Score"]
            + 0.30 * adjusted["Permit_Risk_Score"]
            + 0.22 * adjusted["Inflation_Risk_Score"]
            + 0.21 * adjusted["Site_Condition_Risk_Score"]
            + 1.5 * adjusted["Force_Majeure_Flag"]
        ).clip(1, 10)

    if changed_feature == "Complexity_Level":
        # The target generator uses Complexity_Score, not Complexity_Level
        # directly. Complexity_Level therefore has zero direct ground-truth
        # effect when Complexity_Score is held fixed.
        pass

    return adjusted


gt_sample_n = min(5000, len(X_test))
gt_sample = X_test.sample(n=gt_sample_n, random_state=SEED)
gt_base_probability = generator_noise_free_probability(gt_sample)

rng_gt = np.random.default_rng(SEED + 1000)
gt_rows = []

for feature in features:
    perturbed = gt_sample.copy()
    permuted_values = perturbed[feature].to_numpy().copy()
    rng_gt.shuffle(permuted_values)
    perturbed[feature] = permuted_values
    perturbed = preserve_generator_dependencies(perturbed, feature)

    p_perturbed = generator_noise_free_probability(perturbed)
    gt_rows.append({
        "Feature": feature,
        "Generator_Mean_Absolute_Probability_Change": float(
            np.mean(np.abs(gt_base_probability - p_perturbed))
        )
    })

generator_ground_truth_df = (
    pd.DataFrame(gt_rows)
    .sort_values(
        "Generator_Mean_Absolute_Probability_Change",
        ascending=False
    )
    .reset_index(drop=True)
)
generator_ground_truth_df["Generator_Rank"] = np.arange(
    1, len(generator_ground_truth_df) + 1
)

# Direct coefficient/function audit for each model input.
direct_target_terms = {
    "Complexity_Score": "+0.34 * Complexity_Score",
    "Planned_Duration_Days": "+0.015 * (Planned_Duration_Days/10)",
    "Planned_Budget_Million": "+0.020 * log1p(Planned_Budget_Million)",
    "Design_Change_Count": "+0.10 * Design_Change_Count",
    "Change_Request_Count": "+0.09 * Change_Request_Count",
    "Approval_Duration_Days": "+0.012 * Approval_Duration_Days",
    "Procurement_Lead_Time_Days": "+0.012 * Procurement_Lead_Time_Days",
    "Payment_Delay_Days": "+0.018 * Payment_Delay_Days",
    "External_Risk_Score": "+0.11 * External_Risk_Score",
    "Quality_Defect_Rate": "+0.06 * Quality_Defect_Rate",
    "Safety_Incident_Count": "+0.18 * Safety_Incident_Count",
    "Scope_Clarity_Score": "-0.12 * Scope_Clarity_Score",
    "Stakeholder_Communication_Score": "-0.10 * Stakeholder_Communication_Score",
    "Coordination_Score": "-0.09 * Coordination_Score",
    "Contractor_Experience_Score": "-0.08 * Contractor_Experience_Score",
    "Resource_Availability_Score": "-0.08 * Resource_Availability_Score",
    "Labor_Productivity_Score": "-0.07 * Labor_Productivity_Score",
    "Equipment_Availability_Score": "-0.06 * Equipment_Availability_Score",
    "Supplier_Reliability_Score": "-0.06 * Supplier_Reliability_Score",
    "Contractor_Financial_Stability": "-0.06 * Contractor_Financial_Stability",
    "Owner_Decision_Speed_Score": "-0.05 * Owner_Decision_Speed_Score",
    "Schedule_Buffer_Percent": "-0.035 * Schedule_Buffer_Percent",
    "BIM_Adoption_Level": "-0.18 * BIM_Adoption_Level",
    "AI_Tools_Adoption_Level": "-0.12 * AI_Tools_Adoption_Level",
}

rule_features = [
    {"Authorities_Involved", "Approval_Duration_Days"},
    {"Design_Change_Count", "Scope_Clarity_Score"},
    {"Stakeholder_Communication_Score", "Coordination_Score"},
    {"Resource_Availability_Score", "Labor_Productivity_Score"},
    {"Supplier_Reliability_Score", "Procurement_Lead_Time_Days"},
    {"Contractor_Financial_Stability", "Payment_Delay_Days"},
    {"External_Risk_Score", "Force_Majeure_Flag"},
    {"Schedule_Buffer_Percent", "Complexity_Score"},
    {"BIM_Adoption_Level", "Coordination_Score"},
    {"AI_Tools_Adoption_Level", "Project_Manager_Experience_Years"},
    {"Schedule_Buffer_Percent", "Scope_Clarity_Score"},
]

generator_ground_truth_df["Direct_Target_Term"] = (
    generator_ground_truth_df["Feature"]
    .map(direct_target_terms)
    .fillna("No direct continuous target coefficient")
)
generator_ground_truth_df["Interaction_Rule_Count"] = (
    generator_ground_truth_df["Feature"]
    .map(lambda f: sum(f in feature_set for feature_set in rule_features))
)

# Merge generator importance with the two model-explanation methods.
xai_vs_generator_df = (
    generator_ground_truth_df
    .merge(
        perm_df[["Feature", "Importance_Mean"]]
        .rename(columns={"Importance_Mean": "Model_Permutation_Importance"}),
        on="Feature",
        how="left"
    )
)

if shap_available and not grouped_shap_df.empty:
    xai_vs_generator_df = xai_vs_generator_df.merge(
        grouped_shap_df[
            ["Main_Feature", "Mean_ABS_SHAP"]
        ].rename(columns={
            "Main_Feature": "Feature",
            "Mean_ABS_SHAP": "Model_Mean_ABS_SHAP"
        }),
        on="Feature",
        how="left"
    )
else:
    xai_vs_generator_df["Model_Mean_ABS_SHAP"] = np.nan

xai_vs_generator_df["Permutation_Rank"] = (
    xai_vs_generator_df["Model_Permutation_Importance"]
    .rank(ascending=False, method="average")
)
xai_vs_generator_df["SHAP_Rank"] = (
    xai_vs_generator_df["Model_Mean_ABS_SHAP"]
    .rank(ascending=False, method="average")
)

rank_corr_perm_generator = (
    xai_vs_generator_df[
        ["Generator_Rank", "Permutation_Rank"]
    ].dropna().corr(method="spearman").iloc[0, 1]
)

rank_corr_shap_generator = (
    xai_vs_generator_df[
        ["Generator_Rank", "SHAP_Rank"]
    ].dropna().corr(method="spearman").iloc[0, 1]
    if xai_vs_generator_df["SHAP_Rank"].notna().any()
    else np.nan
)

top_k = 10
top_generator = set(
    xai_vs_generator_df.nsmallest(top_k, "Generator_Rank")["Feature"]
)
top_perm = set(
    xai_vs_generator_df.nsmallest(top_k, "Permutation_Rank")["Feature"]
)
top_shap = set(
    xai_vs_generator_df.nsmallest(top_k, "SHAP_Rank")["Feature"]
) if xai_vs_generator_df["SHAP_Rank"].notna().any() else set()

generator_agreement_summary_df = pd.DataFrame([{
    "Compared_Features": len(xai_vs_generator_df),
    "Generator_vs_Permutation_Spearman": rank_corr_perm_generator,
    "Generator_vs_SHAP_Spearman": rank_corr_shap_generator,
    "Top10_Generator_Permutation_Overlap": len(top_generator & top_perm),
    "Top10_Generator_SHAP_Overlap": len(top_generator & top_shap) if top_shap else np.nan,
    "Ground_Truth_Definition": (
        "Mean absolute change in noise-free generator probability after "
        "single-feature permutation with deterministic dependencies preserved"
    )
}])

display(xai_vs_generator_df.sort_values("Generator_Rank").head(20))
display(generator_agreement_summary_df)

xai_vs_generator_df.to_csv(
    RESULTS_DIR / "FINAL_xai_vs_synthetic_generator_ground_truth.csv",
    index=False
)
generator_agreement_summary_df.to_csv(
    RESULTS_DIR / "FINAL_xai_vs_generator_agreement_summary.csv",
    index=False
)
generator_ground_truth_df.to_csv(
    RESULTS_DIR / "FINAL_synthetic_generator_ground_truth_importance.csv",
    index=False
)

plot_gt = (
    generator_ground_truth_df.head(15)
    .sort_values("Generator_Mean_Absolute_Probability_Change")
)
ax = plot_gt.plot(
    x="Feature",
    y="Generator_Mean_Absolute_Probability_Change",
    kind="barh",
    figsize=(10, 7),
    legend=False
)
ax.set_title("Synthetic Generator Ground-Truth Sensitivity")
ax.set_xlabel("Mean absolute change in noise-free generator probability")
plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "FINAL_synthetic_generator_ground_truth_importance.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 7. Local SHAP Explanation of One Held-Out High-Risk Synthetic Project

A held-out project with a high model-predicted probability is selected for a local SHAP explanation. The signed SHAP values show which features pushed the model prediction upward or downward relative to its baseline expectation.

This is a **prediction explanation**, not an intervention analysis.


In [ ]:
# =========================
# 7.1 Select a Held-Out High-Risk Synthetic Project
# =========================
test_probabilities = pd.Series(
    y_proba_test,
    index=X_test.index,
    name="Predicted_Delay_Probability"
)

selected_idx = test_probabilities.idxmax()
selected_project = X_test.loc[[selected_idx]].copy()
selected_probability = float(test_probabilities.loc[selected_idx])

selected_project_id = (
    df.loc[selected_idx, "Project_ID"]
    if "Project_ID" in df.columns
    else f"Index_{selected_idx}"
)

print("Selected synthetic project:", selected_project_id)
print("Held-out predicted delay probability:", round(selected_probability, 6))


In [ ]:
# =========================
# 7.2 Local SHAP Contributions
# =========================
local_shap_df = pd.DataFrame()

if shap_available and shap_explainer is not None:
    selected_transformed = (
        model.named_steps["preprocessor"]
        .transform(selected_project)
    )

    if hasattr(selected_transformed, "toarray"):
        selected_transformed = selected_transformed.toarray()

    if hasattr(model_step, "feature_importances_"):
        selected_raw_shap = shap_explainer.shap_values(
            selected_transformed
        )
        selected_shap_arr = normalize_shap_array(
            selected_raw_shap
        )
    else:
        selected_explanation = shap_explainer(
            selected_transformed
        )
        selected_shap_arr = normalize_shap_array(
            selected_explanation
        )

    local_transformed_df = pd.DataFrame({
        "Transformed_Feature": transformed_names,
        "Main_Feature": [
            map_to_main_feature(name)
            for name in transformed_names
        ],
        "SHAP_Value": selected_shap_arr[0]
    })

    local_shap_df = (
        local_transformed_df
        .groupby("Main_Feature", as_index=False)["SHAP_Value"]
        .sum()
    )

    local_shap_df["ABS_SHAP_Value"] = (
        local_shap_df["SHAP_Value"].abs()
    )

    local_shap_df["Direction_on_Model_Output"] = np.where(
        local_shap_df["SHAP_Value"] > 0,
        "Pushes prediction higher",
        np.where(
            local_shap_df["SHAP_Value"] < 0,
            "Pushes prediction lower",
            "Neutral"
        )
    )

    local_shap_df = local_shap_df.sort_values(
        "ABS_SHAP_Value",
        ascending=False
    ).reset_index(drop=True)

    local_shap_df.insert(
        0,
        "Project_ID",
        selected_project_id
    )

    display(local_shap_df.head(20))

    plot_local = (
        local_shap_df.head(15)
        .sort_values("SHAP_Value")
    )

    ax = plot_local.plot(
        x="Main_Feature",
        y="SHAP_Value",
        kind="barh",
        figsize=(10, 7),
        legend=False
    )

    ax.set_title(
        "Local SHAP Contributions for a Held-Out High-Risk Synthetic Project"
    )
    ax.set_xlabel("Signed SHAP contribution to model output")

    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR / "FINAL_xai_local_shap_explanation.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    local_shap_df.to_csv(
        RESULTS_DIR / "FINAL_xai_local_shap_explanation.csv",
        index=False
    )

else:
    print("Local SHAP explanation skipped because SHAP was unavailable.")


## 8. Dependency-Aware One-at-a-Time Scenario Sensitivity

This section retains a local sensitivity analysis but separates it from SHAP. Each scenario changes one management-related feature or one logically linked feature group and recalculates the model-predicted probability.

The output is **not causal** and should not be described as a proven risk reduction. It shows only how the trained model responds to the predefined hypothetical change.

Derived relationships are preserved. For example, changing the number of authorities or the days per authority also recalculates `Approval_Duration_Days`.


In [ ]:
# =========================
# 8.1 Local Scenario-Sensitivity Helpers
# =========================
def predict_delay_probability(one_row_df):
    return float(
        positive_class_probability(
            model,
            one_row_df[features]
        )[0]
    )


def apply_local_sensitivity_scenario(project, scenario_name):
    adjusted = project.copy()

    score_increase_features = {
        "Scope_Clarity_Score",
        "Stakeholder_Communication_Score",
        "Coordination_Score",
        "Contractor_Experience_Score",
        "Resource_Availability_Score",
        "Labor_Productivity_Score",
        "Equipment_Availability_Score",
        "Supplier_Reliability_Score",
        "Contractor_Financial_Stability",
        "Owner_Decision_Speed_Score"
    }

    if scenario_name in score_increase_features:
        adjusted.loc[:, scenario_name] = np.minimum(
            adjusted[scenario_name].astype(float) + 1.0,
            10.0
        )

    elif scenario_name == "BIM_Adoption_Level":
        adjusted.loc[:, scenario_name] = np.minimum(
            adjusted[scenario_name].astype(int) + 1,
            3
        )

    elif scenario_name == "AI_Tools_Adoption_Level":
        adjusted.loc[:, scenario_name] = np.minimum(
            adjusted[scenario_name].astype(int) + 1,
            3
        )

    elif scenario_name == "Schedule_Buffer_Percent":
        adjusted.loc[:, scenario_name] = np.minimum(
            adjusted[scenario_name].astype(float) + 1.0,
            25.0
        )

    elif scenario_name == "Planned_Budget_Million":
        adjusted.loc[:, scenario_name] = (
            adjusted[scenario_name].astype(float) * 1.05
        )

    elif scenario_name == "Planned_Duration_Days":
        adjusted.loc[:, scenario_name] = np.rint(
            adjusted[scenario_name].astype(float) * 1.05
        ).astype(int)

    elif scenario_name == "Change_Request_Count":
        adjusted.loc[:, scenario_name] = np.maximum(
            adjusted[scenario_name].astype(int) - 1,
            0
        )

    elif scenario_name == "Design_Change_Count":
        adjusted.loc[:, scenario_name] = np.maximum(
            adjusted[scenario_name].astype(int) - 1,
            0
        )

    elif scenario_name == "Procurement_Lead_Time_Days":
        adjusted.loc[:, scenario_name] = np.maximum(
            adjusted[scenario_name].astype(int) - 7,
            5
        )

    elif scenario_name == "Payment_Delay_Days":
        adjusted.loc[:, scenario_name] = np.maximum(
            adjusted[scenario_name].astype(int) - 7,
            0
        )

    elif scenario_name == "Authorities_Involved":
        adjusted.loc[:, scenario_name] = np.maximum(
            adjusted[scenario_name].astype(int) - 1,
            1
        )

        adjusted.loc[:, "Approval_Duration_Days"] = (
            adjusted["Authorities_Involved"].astype(int)
            * adjusted["Approval_Per_Authority_Days"].astype(int)
        )

    elif scenario_name == "Approval_Per_Authority_Days":
        adjusted.loc[:, scenario_name] = np.maximum(
            adjusted[scenario_name].astype(int) - 2,
            1
        )

        adjusted.loc[:, "Approval_Duration_Days"] = (
            adjusted["Authorities_Involved"].astype(int)
            * adjusted["Approval_Per_Authority_Days"].astype(int)
        )

    else:
        raise ValueError(f"Unknown local scenario: {scenario_name}")

    return adjusted


In [ ]:
# =========================
# 8.2 Run One-at-a-Time Local Sensitivity
# =========================
local_scenario_features = [
    "Scope_Clarity_Score",
    "Stakeholder_Communication_Score",
    "Coordination_Score",
    "Contractor_Experience_Score",
    "Resource_Availability_Score",
    "Labor_Productivity_Score",
    "Equipment_Availability_Score",
    "Supplier_Reliability_Score",
    "Contractor_Financial_Stability",
    "Owner_Decision_Speed_Score",
    "Change_Request_Count",
    "Design_Change_Count",
    "Authorities_Involved",
    "Approval_Per_Authority_Days",
    "Procurement_Lead_Time_Days",
    "Payment_Delay_Days",
    "BIM_Adoption_Level",
    "AI_Tools_Adoption_Level",
    "Schedule_Buffer_Percent",
    "Planned_Budget_Million",
    "Planned_Duration_Days"
]

base_prob = predict_delay_probability(selected_project)
sensitivity_rows = []

for scenario_feature in local_scenario_features:
    if scenario_feature not in selected_project.columns:
        continue

    adjusted = apply_local_sensitivity_scenario(
        selected_project,
        scenario_feature
    )

    new_prob = predict_delay_probability(adjusted)

    before_value = selected_project.iloc[0][scenario_feature]
    after_value = adjusted.iloc[0][scenario_feature]

    sensitivity_rows.append({
        "Project_ID": selected_project_id,
        "Scenario_Feature": scenario_feature,
        "Before": before_value,
        "After": after_value,
        "Base_Probability": base_prob,
        "Scenario_Probability": new_prob,
        "Predicted_Probability_Change": new_prob - base_prob,
        "Predicted_Probability_Reduction": base_prob - new_prob
    })

local_sensitivity_df = pd.DataFrame(
    sensitivity_rows
).sort_values(
    "Predicted_Probability_Reduction",
    ascending=False
).reset_index(drop=True)

display(local_sensitivity_df.head(25))

plot_sensitivity = (
    local_sensitivity_df.head(15)
    .sort_values("Predicted_Probability_Reduction")
)

ax = plot_sensitivity.plot(
    x="Scenario_Feature",
    y="Predicted_Probability_Reduction",
    kind="barh",
    figsize=(10, 7),
    legend=False
)

ax.set_title(
    "One-at-a-Time Model Sensitivity for a Held-Out Synthetic Project"
)
ax.set_xlabel("Model-predicted delay probability reduction")

plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "FINAL_xai_local_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

local_sensitivity_df.to_csv(
    RESULTS_DIR / "FINAL_xai_local_sensitivity.csv",
    index=False
)

print(
    "Base model-predicted probability for selected synthetic project:",
    round(base_prob, 6)
)


## 9. Reporting Guidance

Use the outputs from this notebook with the following interpretation:

- **Permutation importance** identifies original input features on which held-out model F1 depends.
- **SHAP importance** identifies features with the largest average contribution magnitude to XGBoost predictions.
- **Synthetic-generator ground-truth sensitivity** measures how strongly the known, noise-free simulation mechanism responds when each feature is perturbed.
- Agreement of SHAP/permutation with generator ground truth is evidence of **internal explanation fidelity to the simulation**, not empirical truth about real projects.
- **Local SHAP** explains one model prediction and may be moved to supplementary material.
- **Local scenario sensitivity** shows model response to predefined hypothetical changes and is not causal.

Suitable wording is: *“Within the synthetic benchmark, the model explanations were compared with the known generator mechanism to assess internal explanation fidelity.”*

Avoid wording such as *“SHAP proved that these variables cause project delay”* or *“the model discovered real-world delay drivers.”*


In [ ]:
# =========================
# 10. Reproducibility Metadata
# =========================
xai_metadata = {
    "seed": SEED,
    "selected_model": best_model_name,
    "dataset_records": int(len(df)),
    "held_out_test_records": int(len(X_test)),
    "baseline_seed": baseline_seed,
    "baseline_test_fraction": test_fraction,
    "permutation_sample_n": int(min(PERMUTATION_SAMPLE_N, len(X_test))),
    "permutation_repeats": PERMUTATION_REPEATS,
    "permutation_scoring": "f1",
    "shap_sample_n": int(min(SHAP_SAMPLE_N, len(X_test))),
    "shap_available": bool(shap_available),
    "shap_version": shap_version,
    "selected_local_project_id": str(selected_project_id),
    "selected_local_project_probability": float(selected_probability),
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "generator_ground_truth_comparison_file": str(
        RESULTS_DIR / "FINAL_xai_vs_synthetic_generator_ground_truth.csv"
    ),
    "generator_ground_truth_summary_file": str(
        RESULTS_DIR / "FINAL_xai_vs_generator_agreement_summary.csv"
    ),
    "interpretation_scope": (
        "XAI results explain the trained model within a synthetic "
        "proof-of-concept environment. They are not causal findings "
        "and require real-world validation."
    )
}

with open(
    RESULTS_DIR / "FINAL_xai_reproducibility_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(xai_metadata, f, indent=2)

print(json.dumps(xai_metadata, indent=2))
